In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

plant_leaves_super_resolution_challenge_path = kagglehub.competition_download('plant-leaves-super-resolution-challenge')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Score : 16.3994369 (4.7)
import os
from pathlib import Path

BASE = Path('/kaggle/input')

print("=" * 60)
print("Contents of /kaggle/input:")
print("=" * 60)

for root, dirs, files in os.walk(BASE):
    level = root.replace(str(BASE), '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:  # limit depth shown for files
        sub_indent = '  ' * (level + 1)
        for f in files[:5]:  # show only first 5 files per dir
            print(f"{sub_indent}{f}")
        if len(files) > 5:
            print(f"{sub_indent}... ({len(files)} files total)")

In [ ]:
# Score : 16.3994369 (4.7)
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import torchvision.models as tvm

# ============================================================
# Reproducibility
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# ============================================================
# Paths
# ============================================================
BASE = Path('/kaggle/input')
# Auto-detect dataset folder
candidates = list(BASE.glob('*/train_High_Resolution'))
if len(candidates) == 0:
    candidates = list(BASE.glob('**/train_High_Resolution'))
DATA_ROOT = candidates[0].parent
print("Data root:", DATA_ROOT)

HR_DIR = DATA_ROOT / 'train_High_Resolution'
LR_DIR = DATA_ROOT / 'train_Low_Resolution'
TEST_DIR = DATA_ROOT / 'test_Low_Resolution'

# Try finding VGG19 weights
vgg_candidates = list(BASE.glob('**/vgg19*.pth'))
VGG_PATH = vgg_candidates[0] if vgg_candidates else None
print("VGG19 path:", VGG_PATH)

OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True, parents=True)

# ============================================================
# Dataset
# ============================================================
class SRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, filenames, augment=False):
        self.lr_dir = Path(lr_dir)
        self.hr_dir = Path(hr_dir)
        self.filenames = filenames
        self.augment = augment

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fn = self.filenames[idx]
        lr = Image.open(self.lr_dir / fn).convert('RGB')
        hr = Image.open(self.hr_dir / fn).convert('RGB')
        lr = np.array(lr, dtype=np.float32) / 255.0
        hr = np.array(hr, dtype=np.float32) / 255.0

        if self.augment:
            # Random horizontal flip
            if random.random() < 0.5:
                lr = lr[:, ::-1, :].copy()
                hr = hr[:, ::-1, :].copy()
            # Random vertical flip
            if random.random() < 0.5:
                lr = lr[::-1, :, :].copy()
                hr = hr[::-1, :, :].copy()
            # Random 90-deg rotation
            k = random.randint(0, 3)
            if k > 0:
                lr = np.rot90(lr, k, axes=(0, 1)).copy()
                hr = np.rot90(hr, k, axes=(0, 1)).copy()

        lr = torch.from_numpy(lr).permute(2, 0, 1).float()
        hr = torch.from_numpy(hr).permute(2, 0, 1).float()
        return lr, hr, fn


class TestDataset(Dataset):
    def __init__(self, lr_dir, filenames):
        self.lr_dir = Path(lr_dir)
        self.filenames = filenames

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fn = self.filenames[idx]
        lr = Image.open(self.lr_dir / fn).convert('RGB')
        lr = np.array(lr, dtype=np.float32) / 255.0
        lr = torch.from_numpy(lr).permute(2, 0, 1).float()
        return lr, fn


# ============================================================
# Generator: RRDB-lite / EDSR-style
# ============================================================
class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(ch, ch, 3, 1, 1)

    def forward(self, x):
        r = F.relu(self.conv1(x), inplace=True)
        r = self.conv2(r)
        return x + r * 0.1


class UpsampleBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch * 4, 3, 1, 1)
        self.ps = nn.PixelShuffle(2)

    def forward(self, x):
        return F.relu(self.ps(self.conv(x)), inplace=True)


class Generator(nn.Module):
    def __init__(self, num_feats=64, num_blocks=16):
        super().__init__()
        self.head = nn.Conv2d(3, num_feats, 3, 1, 1)
        self.body = nn.Sequential(*[ResidualBlock(num_feats) for _ in range(num_blocks)])
        self.body_conv = nn.Conv2d(num_feats, num_feats, 3, 1, 1)
        self.up1 = UpsampleBlock(num_feats)
        self.up2 = UpsampleBlock(num_feats)
        self.tail = nn.Sequential(
            nn.Conv2d(num_feats, num_feats, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(num_feats, 3, 3, 1, 1)
        )

    def forward(self, x):
        # Bicubic base to help preserve spatial content (residual learning)
        base = F.interpolate(x, scale_factor=4, mode='bicubic', align_corners=False)
        f = self.head(x)
        res = self.body_conv(self.body(f)) + f
        u = self.up1(res)
        u = self.up2(u)
        out = self.tail(u)
        return torch.clamp(base + out, 0.0, 1.0)


# ============================================================
# Discriminator
# ============================================================
class Discriminator(nn.Module):
    def __init__(self, in_ch=3):
        super().__init__()
        def block(ic, oc, s=2, bn=True):
            layers = [nn.Conv2d(ic, oc, 4, s, 1)]
            if bn:
                layers.append(nn.BatchNorm2d(oc))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.net = nn.Sequential(
            *block(in_ch, 64, 2, bn=False),
            *block(64, 128, 2),
            *block(128, 256, 2),
            *block(256, 512, 2),
            nn.Conv2d(512, 1, 4, 1, 1)
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# VGG19 Perceptual Loss (only if weights available)
# ============================================================
class VGGPerceptual(nn.Module):
    def __init__(self, weights_path):
        super().__init__()
        vgg = tvm.vgg19(weights=None)
        try:
            sd = torch.load(weights_path, map_location='cpu')
            vgg.load_state_dict(sd)
            print("Loaded VGG19 weights.")
            self.ok = True
        except Exception as e:
            print("Failed to load VGG19:", e)
            self.ok = False
        # Use features up to relu5_4 index 35 or relu2_2 at index 8
        self.feat = nn.Sequential(*list(vgg.features.children())[:16]).eval()
        for p in self.feat.parameters():
            p.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x, y):
        x = (x - self.mean) / self.std
        y = (y - self.mean) / self.std
        return F.l1_loss(self.feat(x), self.feat(y))


# ============================================================
# Training
# ============================================================
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.kaiming_normal_(m.weight, a=0.2, nonlinearity='leaky_relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)


def psnr(pred, target):
    mse = F.mse_loss(pred, target)
    if mse.item() == 0:
        return 100.0
    return 10 * torch.log10(1.0 / mse).item()


def train():
    all_files = sorted([f for f in os.listdir(HR_DIR) if f.endswith('.png')])
    print("Total training files:", len(all_files))

    # Deterministic split
    rng = random.Random(SEED)
    shuffled = all_files.copy()
    rng.shuffle(shuffled)
    n_val = max(50, int(0.05 * len(shuffled)))
    val_files = shuffled[:n_val]
    train_files = shuffled[n_val:]

    train_ds = SRDataset(LR_DIR, HR_DIR, train_files, augment=True)
    val_ds = SRDataset(LR_DIR, HR_DIR, val_files, augment=False)

    BATCH = 16
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

    G = Generator(num_feats=64, num_blocks=16).to(device)
    D = Discriminator().to(device)
    G.apply(init_weights)
    D.apply(init_weights)

    vgg_loss = None
    if VGG_PATH is not None and VGG_PATH.exists():
        try:
            vgg_loss = VGGPerceptual(VGG_PATH).to(device)
            if not vgg_loss.ok:
                vgg_loss = None
        except Exception as e:
            print("VGG init failed:", e)
            vgg_loss = None

    opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.9, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=1e-4, betas=(0.9, 0.999))

    bce = nn.BCEWithLogitsLoss()
    l1 = nn.L1Loss()

    EPOCHS = 60
    PRETRAIN_EPOCHS = 10  # L1 only pretrain

    # Loss weights - L1 dominant
    ALPHA = 1.0    # L1
    BETA = 5e-4    # adv (very small to protect MAE)
    GAMMA = 0.01   # perceptual

    scaler_G = torch.cuda.amp.GradScaler()
    scaler_D = torch.cuda.amp.GradScaler()

    best_val_mae = float('inf')
    ckpt_path = OUT_DIR / 'generator_best.pth'

    for epoch in range(EPOCHS):
        G.train(); D.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        g_loss_acc, d_loss_acc, l1_acc = 0, 0, 0
        n = 0

        for lr_img, hr_img, _ in pbar:
            lr_img = lr_img.to(device, non_blocking=True)
            hr_img = hr_img.to(device, non_blocking=True)

            use_gan = epoch >= PRETRAIN_EPOCHS

            # ---- Train Discriminator ----
            if use_gan:
                opt_D.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast():
                    with torch.no_grad():
                        fake = G(lr_img)
                    d_real = D(hr_img)
                    d_fake = D(fake.detach())
                    real_labels = torch.ones_like(d_real)
                    fake_labels = torch.zeros_like(d_fake)
                    loss_D = 0.5 * (bce(d_real, real_labels) + bce(d_fake, fake_labels))
                scaler_D.scale(loss_D).backward()
                scaler_D.step(opt_D)
                scaler_D.update()
                d_loss_acc += loss_D.item()

            # ---- Train Generator ----
            opt_G.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                fake = G(lr_img)
                loss_l1 = l1(fake, hr_img)
                loss_total = ALPHA * loss_l1
                if use_gan:
                    d_fake = D(fake)
                    loss_adv = bce(d_fake, torch.ones_like(d_fake))
                    loss_total = loss_total + BETA * loss_adv
                if vgg_loss is not None:
                    loss_p = vgg_loss(fake, hr_img)
                    loss_total = loss_total + GAMMA * loss_p

            scaler_G.scale(loss_total).backward()
            scaler_G.step(opt_G)
            scaler_G.update()

            g_loss_acc += loss_total.item()
            l1_acc += loss_l1.item()
            n += 1
            pbar.set_postfix(l1=f"{l1_acc/n:.4f}", g=f"{g_loss_acc/n:.4f}", d=f"{d_loss_acc/max(n,1):.4f}")

        # ---- Validation ----
        G.eval()
        val_mae, val_psnr, vn = 0, 0, 0
        with torch.no_grad():
            for lr_img, hr_img, _ in val_loader:
                lr_img = lr_img.to(device)
                hr_img = hr_img.to(device)
                fake = G(lr_img)
                val_mae += F.l1_loss(fake * 255, hr_img * 255).item()
                val_psnr += psnr(fake, hr_img)
                vn += 1
        val_mae /= vn
        val_psnr /= vn
        print(f"[Epoch {epoch+1}] Val MAE (0-255): {val_mae:.4f} | PSNR: {val_psnr:.3f}")

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            torch.save(G.state_dict(), ckpt_path)
            print(f"  Saved best model (MAE={val_mae:.4f})")

    # Also save final
    torch.save(G.state_dict(), OUT_DIR / 'generator_final.pth')
    return ckpt_path


ckpt_path = train()

# ============================================================
# Inference
# ============================================================
def inference(ckpt_path):
    G = Generator(num_feats=64, num_blocks=16).to(device)
    G.load_state_dict(torch.load(ckpt_path, map_location=device))
    G.eval()

    test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith('.png')])
    print("Test files:", len(test_files))

    ds = TestDataset(TEST_DIR, test_files)
    loader = DataLoader(ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

    rows = []
    with torch.no_grad():
        for lr_img, fns in tqdm(loader, desc="Inference"):
            lr_img = lr_img.to(device)
            # TTA: horizontal flip averaging (helps MAE slightly)
            out1 = G(lr_img)
            out2 = torch.flip(G(torch.flip(lr_img, dims=[3])), dims=[3])
            out = (out1 + out2) / 2.0
            out = torch.clamp(out, 0.0, 1.0)
            out_np = (out.cpu().numpy() * 255.0).round().astype(np.uint8)
            # Shape: (B, 3, 128, 128) -> per image (128, 128, 3)
            for i, fn in enumerate(fns):
                img = out_np[i].transpose(1, 2, 0)  # H, W, C
                flat = img.reshape(-1)  # row-major, per-pixel RGB interleaved
                pixel_str = ' '.join(flat.astype(str).tolist())
                rows.append({'Id': fn, 'Pixels': pixel_str})

    sub = pd.DataFrame(rows)
    sub_path = OUT_DIR / 'submission.csv'
    sub.to_csv(sub_path, index=False)
    print("Saved:", sub_path, "rows:", len(sub))
    return sub_path


inference(ckpt_path)
print("Done.")

In [ ]:
# ============================================================
# Super-Resolution cGAN: 32×32 → 128×128
# Kaggle Competition Solution - FULLY FIXED VERSION
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ============================================================
# 1. REPRODUCIBILITY
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ============================================================
# 2. CONFIGURATION
# ============================================================
class Config:
    LR_SIZE  = 32
    HR_SIZE  = 128
    SCALE    = 4
    CHANNELS = 3

    BATCH_SIZE  = 16
    NUM_EPOCHS  = 150
    LR_G        = 1e-4
    LR_D        = 1e-4
    BETA1       = 0.9
    BETA2       = 0.999
    VAL_SPLIT   = 0.1
    NUM_WORKERS = 2

    # Loss weights — L1 dominant so MAE is minimised
    LAMBDA_L1   = 100.0
    LAMBDA_ADV  = 1.0
    LAMBDA_PERC = 10.0

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    SAVE_EVERY      = 10
    USE_PERCEPTUAL  = True
    MIXED_PRECISION = True

    CHECKPOINT_DIR  = "/kaggle/working/checkpoints"
    OUTPUT_DIR      = "/kaggle/working/outputs"
    SUBMISSION_PATH = "/kaggle/working/submission.csv"

cfg = Config()
os.makedirs(cfg.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(cfg.OUTPUT_DIR,     exist_ok=True)

# ============================================================
# 3. PATH RESOLUTION
# ============================================================
def find_image_files(directory):
    """Return sorted list of image Paths inside directory."""
    directory = Path(directory)
    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {directory}")
    exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif', '.webp'}
    files = sorted([f for f in directory.iterdir()
                    if f.is_file() and f.suffix.lower() in exts])
    return files


def resolve_paths():
    base = Path("/kaggle/input/competitions/plant-leaves-super-resolution-challenge")
    print("Scanning /kaggle/input …")
    if base.exists():
        for d in sorted(base.iterdir()):
            if d.is_dir():
                subs = [s for s in d.iterdir() if s.is_dir()]
                print(f"  {d.name}/  →  {[s.name for s in subs]}")

    hr_names   = ["train_High_Resolution", "train_high_resolution",
                  "Train_High_Resolution", "HR", "hr", "train_hr"]
    lr_names   = ["train_Low_Resolution",  "train_low_resolution",
                  "Train_Low_Resolution",  "LR", "lr", "train_lr"]
    test_names = ["test_Low_Resolution",   "test_low_resolution",
                  "Test_Low_Resolution",   "test_lr", "test"]
    vgg_names  = ["vgg19_weights.pth", "vgg19.pth", "vgg_weights.pth"]

    def find_dir(candidates):
        if not base.exists():
            return None
        for sub in base.iterdir():
            if not sub.is_dir():
                continue
            for c in candidates:
                p = sub / c
                if p.is_dir():
                    return p
        for c in candidates:
            for m in base.rglob(c):
                if m.is_dir():
                    return m
        return None

    def find_file(candidates):
        if not base.exists():
            return None
        for sub in base.iterdir():
            if not sub.is_dir():
                continue
            for c in candidates:
                p = sub / c
                if p.is_file():
                    return p
        for c in candidates:
            for m in base.rglob(c):
                if m.is_file():
                    return m
        return None

    hr   = find_dir(hr_names)
    lr   = find_dir(lr_names)
    test = find_dir(test_names)
    vgg  = find_file(vgg_names)

    print(f"\nResolved:")
    print(f"  Train HR : {hr}")
    print(f"  Train LR : {lr}")
    print(f"  Test  LR : {test}")
    print(f"  VGG19    : {vgg}")
    return hr, lr, test, vgg


TRAIN_HR_DIR, TRAIN_LR_DIR, TEST_LR_DIR, VGG_WEIGHTS_PATH = resolve_paths()

assert TRAIN_HR_DIR is not None, "Could not find Train HR directory!"
assert TRAIN_LR_DIR is not None, "Could not find Train LR directory!"
assert TEST_LR_DIR  is not None, "Could not find Test  LR directory!"

train_hr_files = find_image_files(TRAIN_HR_DIR)
train_lr_files = find_image_files(TRAIN_LR_DIR)
test_lr_files  = find_image_files(TEST_LR_DIR)

print(f"\nFiles found — HR: {len(train_hr_files)}  "
      f"LR: {len(train_lr_files)}  Test: {len(test_lr_files)}")
assert len(train_hr_files) > 0
assert len(train_lr_files) > 0
assert len(test_lr_files)  > 0
assert len(train_hr_files) == len(train_lr_files), \
    f"HR/LR count mismatch: {len(train_hr_files)} vs {len(train_lr_files)}"

print(f"Device : {cfg.DEVICE}")

# ============================================================
# 4. DATASET
# ============================================================
class SRDataset(Dataset):
    def __init__(self, lr_files, hr_files=None, is_test=False, augment=True):
        self.lr_files = [Path(f) for f in lr_files]
        self.hr_files = [Path(f) for f in hr_files] if hr_files else None
        self.is_test  = is_test
        self.augment  = augment and not is_test

        self.lr_tf = transforms.Compose([
            transforms.Resize((cfg.LR_SIZE, cfg.LR_SIZE),
                              interpolation=transforms.InterpolationMode.BICUBIC,
                              antialias=True),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])
        self.hr_tf = transforms.Compose([
            transforms.Resize((cfg.HR_SIZE, cfg.HR_SIZE),
                              interpolation=transforms.InterpolationMode.BICUBIC,
                              antialias=True),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])

    def __len__(self):
        return len(self.lr_files)

    @staticmethod
    def _aug(lr, hr):
        if random.random() > 0.5:
            lr = lr.transpose(Image.FLIP_LEFT_RIGHT)
            hr = hr.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() > 0.5:
            lr = lr.transpose(Image.FLIP_TOP_BOTTOM)
            hr = hr.transpose(Image.FLIP_TOP_BOTTOM)
        k = random.randint(0, 3)
        if k:
            lr = lr.rotate(90 * k)
            hr = hr.rotate(90 * k)
        return lr, hr

    def __getitem__(self, idx):
        lr_img = Image.open(self.lr_files[idx]).convert("RGB")
        if self.is_test:
            return self.lr_tf(lr_img), self.lr_files[idx].name

        hr_img = Image.open(self.hr_files[idx]).convert("RGB")
        if self.augment:
            lr_img, hr_img = self._aug(lr_img, hr_img)
        return self.lr_tf(lr_img), self.hr_tf(hr_img)


# ============================================================
# 5. MODEL — FIXED ResidualDenseBlock
# ============================================================

class ResidualDenseBlock(nn.Module):
    """
    Dense block where every conv sees ALL previous feature maps.

    Channel accounting (channels=C, growth_rate=G):
        input  : C
        conv1 in= C       → out= G
        conv2 in= C+G     → out= G
        conv3 in= C+2G    → out= G
        conv4 in= C+3G    → out= G
        conv5 in= C+4G    → out= C   (bottleneck back to C)
    """
    def __init__(self, channels: int = 64, growth_rate: int = 32):
        super().__init__()
        C, G = channels, growth_rate
        self.conv1 = nn.Conv2d(C,       G, 3, 1, 1)
        self.conv2 = nn.Conv2d(C + G,   G, 3, 1, 1)
        self.conv3 = nn.Conv2d(C + 2*G, G, 3, 1, 1)
        self.conv4 = nn.Conv2d(C + 3*G, G, 3, 1, 1)
        self.conv5 = nn.Conv2d(C + 4*G, C, 3, 1, 1)   # ← back to C channels
        self.act   = nn.LeakyReLU(0.2, inplace=True)
        self.beta  = 0.2
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=0.2)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # x  : (B, C, H, W)
        x1 = self.act(self.conv1(x))                              # (B, G, H, W)
        x2 = self.act(self.conv2(torch.cat([x,  x1],     dim=1))) # (B, G, H, W)
        x3 = self.act(self.conv3(torch.cat([x,  x1, x2], dim=1)))
        x4 = self.act(self.conv4(torch.cat([x,  x1, x2, x3], dim=1)))
        x5 =          self.conv5(torch.cat([x,  x1, x2, x3, x4], dim=1))
        return x5 * self.beta + x   # residual, output: (B, C, H, W)


class RRDB(nn.Module):
    """Residual-in-Residual Dense Block."""
    def __init__(self, channels: int = 64, growth_rate: int = 32):
        super().__init__()
        self.rdb1  = ResidualDenseBlock(channels, growth_rate)
        self.rdb2  = ResidualDenseBlock(channels, growth_rate)
        self.rdb3  = ResidualDenseBlock(channels, growth_rate)
        self.beta  = 0.2

    def forward(self, x):
        out = self.rdb3(self.rdb2(self.rdb1(x)))
        return out * self.beta + x   # output: same shape as x


class Generator(nn.Module):
    """
    RRDB Generator  32×32 → 128×128  (scale ×4).

    Flow:
        conv_first → trunk (N × RRDB) → conv_trunk
        → (skip add) → up1 (×2, PixelShuffle) → up2 (×2, PixelShuffle)
        → hr_conv → out_conv → tanh
    All intermediate tensors stay at `nf` channels. ✓
    """
    def __init__(self, in_ch=3, out_ch=3, nf=64, nb=16, gc=32):
        super().__init__()
        self.conv_first  = nn.Conv2d(in_ch, nf, 3, 1, 1)
        self.trunk       = nn.Sequential(*[RRDB(nf, gc) for _ in range(nb)])
        self.conv_trunk  = nn.Conv2d(nf, nf, 3, 1, 1)

        # ×2 via PixelShuffle: nf → nf*4 → (PixelShuffle) → nf
        self.up1 = nn.Sequential(
            nn.Conv2d(nf, nf * 4, 3, 1, 1),
            nn.PixelShuffle(2),            # nf*4 → nf, H*2, W*2
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.up2 = nn.Sequential(
            nn.Conv2d(nf, nf * 4, 3, 1, 1),
            nn.PixelShuffle(2),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.hr_conv  = nn.Sequential(
            nn.Conv2d(nf, nf, 3, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.out_conv = nn.Conv2d(nf, out_ch, 3, 1, 1)
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=0.2)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        # Small init on final layer → output near zero at start
        nn.init.normal_(self.out_conv.weight, 0.0, 0.01)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, x):
        feat  = self.conv_first(x)            # (B, nf, 32, 32)
        trunk = self.conv_trunk(self.trunk(feat))
        feat  = feat + trunk                  # global residual
        feat  = self.up1(feat)               # (B, nf, 64, 64)
        feat  = self.up2(feat)               # (B, nf, 128, 128)
        feat  = self.hr_conv(feat)
        return torch.tanh(self.out_conv(feat))


# ----------------------------------------------------------
# Discriminator (PatchGAN)
# ----------------------------------------------------------
class DiscBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=2, bn=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, stride, 1, bias=not bn)]
        if bn:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class Discriminator(nn.Module):
    def __init__(self, in_ch=3, nf=64):
        super().__init__()
        self.model = nn.Sequential(
            DiscBlock(in_ch,  nf,    stride=2, bn=False),
            DiscBlock(nf,     nf*2,  stride=2, bn=True),
            DiscBlock(nf*2,   nf*4,  stride=2, bn=True),
            DiscBlock(nf*4,   nf*8,  stride=1, bn=True),
            nn.Conv2d(nf*8, 1, 4, 1, 1),
        )
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0.0, 0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight, 1.0, 0.02)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.model(x)


# ============================================================
# 6. PERCEPTUAL LOSS (VGG19 — no pretrained=True)
# ============================================================
class VGG19Features(nn.Module):
    _CFG = [64, 64, 'M', 128, 128, 'M',
            256, 256, 256, 256, 'M',
            512, 512, 512, 512, 'M',
            512, 512, 512, 512, 'M']

    def __init__(self, weight_path=None):
        super().__init__()
        self.features = self._build()
        self._load(weight_path)
        for p in self.parameters():
            p.requires_grad = False

        ch = list(self.features.children())
        self.s1 = nn.Sequential(*ch[:4])    # relu1_2
        self.s2 = nn.Sequential(*ch[4:9])   # relu2_2
        self.s3 = nn.Sequential(*ch[9:18])  # relu3_3

    def _build(self):
        layers, ic = [], 3
        for v in self._CFG:
            if v == 'M':
                layers.append(nn.MaxPool2d(2, 2))
            else:
                layers += [nn.Conv2d(ic, v, 3, padding=1), nn.ReLU(inplace=True)]
                ic = v
        return nn.Sequential(*layers)

    def _load(self, path):
        if path is None or not os.path.exists(path):
            print("VGG19: weight file not found — random weights used."); return
        try:
            sd = torch.load(path, map_location='cpu')
            if isinstance(sd, dict) and 'state_dict' in sd:
                sd = sd['state_dict']
            clean = {}
            for k, v in sd.items():
                for pfx in ('features.', 'module.features.'):
                    if k.startswith(pfx):
                        k = k[len(pfx):]; break
                clean[k] = v
            miss, unexp = self.features.load_state_dict(clean, strict=False)
            print(f"VGG19 weights loaded — missing {len(miss)}, unexpected {len(unexp)}")
        except Exception as e:
            print(f"VGG19 load error: {e} — random weights used.")

    def forward(self, x):
        h1 = self.s1(x); h2 = self.s2(h1); h3 = self.s3(h2)
        return h1, h2, h3


class PerceptualLoss(nn.Module):
    def __init__(self, weight_path=None):
        super().__init__()
        self.vgg = VGG19Features(weight_path)
        self.criterion = nn.L1Loss()
        self.register_buffer('mean', torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))

    def _norm(self, x):
        x = (x + 1.0) / 2.0          # [-1,1] → [0,1]
        return (x - self.mean) / self.std

    def forward(self, fake, real):
        ff = self.vgg(self._norm(fake))
        rf = self.vgg(self._norm(real))
        loss = 0.0
        for w, f, r in zip([1.0, 0.5, 0.25], ff, rf):
            loss += w * self.criterion(f, r.detach())
        return loss


# ============================================================
# 7. GAN LOSS (LSGAN)
# ============================================================
class GANLoss(nn.Module):
    def __init__(self, smooth=0.1):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, is_real: bool):
        tgt = torch.full_like(pred, 1.0 - self.smooth if is_real else self.smooth)
        return F.mse_loss(pred, tgt)


# ============================================================
# 8. UTILITIES
# ============================================================
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.avg = self.sum = self.count = 0.0
    def update(self, v, n=1):
        self.sum += v * n; self.count += n; self.avg = self.sum / self.count


def psnr(fake, real):
    mse = F.mse_loss(fake.detach(), real.detach())
    return 20.0 * torch.log10(torch.tensor(2.0) / torch.sqrt(mse + 1e-8))


def tensor_to_uint8(t):
    """CHW tensor [-1,1] → HWC uint8 numpy."""
    t = t.detach().cpu().float()
    t = ((t + 1.0) / 2.0).clamp(0, 1).permute(1, 2, 0)
    return (t.numpy() * 255.0).round().astype(np.uint8)


def save_ckpt(state, path):
    torch.save(state, path)


def load_ckpt(path, G, D, og, od, device):
    c = torch.load(path, map_location=device)
    G.load_state_dict(c['generator'])
    D.load_state_dict(c['discriminator'])
    og.load_state_dict(c['opt_g'])
    od.load_state_dict(c['opt_d'])
    print(f"Resumed epoch {c['epoch']+1}  best_mae={c.get('val_mae','?'):.4f}")
    return c['epoch']


# ============================================================
# 9. AMP CONTEXT HELPER  (suppresses FutureWarning)
# ============================================================
def amp_autocast(enabled: bool):
    """
    Returns the correct autocast context for the installed PyTorch version.
    torch >= 2.0 : torch.amp.autocast('cuda', ...)
    older        : torch.cuda.amp.autocast(...)
    """
    try:
        return torch.amp.autocast('cuda', enabled=enabled)
    except Exception:
        return torch.cuda.amp.autocast(enabled=enabled)


# ============================================================
# 10. TRAINING
# ============================================================
def build_split(lr_files, hr_files, val_frac=0.1, seed=42):
    n = len(lr_files)
    n_val   = max(1, int(n * val_frac))
    n_train = n - n_val
    assert n_train > 0 and n_val > 0, f"Too few samples: {n}"

    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    tr, va = idx[:n_train], idx[n_train:]

    return ([lr_files[i] for i in tr], [hr_files[i] for i in tr],
            [lr_files[i] for i in va], [hr_files[i] for i in va])


def train():
    print("=" * 60)
    print("SUPER-RESOLUTION cGAN — TRAINING")
    print("=" * 60)

    # ── Split ──────────────────────────────────────────────
    tr_lr, tr_hr, va_lr, va_hr = build_split(
        train_lr_files, train_hr_files, cfg.VAL_SPLIT)
    print(f"Train: {len(tr_lr)}  |  Val: {len(va_lr)}")

    nw = cfg.NUM_WORKERS
    pin = cfg.DEVICE.type == 'cuda'

    train_loader = DataLoader(
        SRDataset(tr_lr, tr_hr, augment=True),
        batch_size=cfg.BATCH_SIZE, shuffle=True,
        num_workers=nw, pin_memory=pin, drop_last=True,
        persistent_workers=(nw > 0),
    )
    val_loader = DataLoader(
        SRDataset(va_lr, va_hr, augment=False),
        batch_size=cfg.BATCH_SIZE, shuffle=False,
        num_workers=nw, pin_memory=pin,
        persistent_workers=(nw > 0),
    )

    # ── Models ─────────────────────────────────────────────
    G = Generator(nf=64, nb=16, gc=32).to(cfg.DEVICE)
    D = Discriminator(nf=64).to(cfg.DEVICE)
    print(f"G params: {sum(p.numel() for p in G.parameters())/1e6:.2f}M  "
          f"D params: {sum(p.numel() for p in D.parameters())/1e6:.2f}M")

    # ── Quick forward-pass sanity check ────────────────────
    with torch.no_grad():
        dummy = torch.zeros(1, 3, cfg.LR_SIZE, cfg.LR_SIZE).to(cfg.DEVICE)
        out   = G(dummy)
        assert out.shape == (1, 3, cfg.HR_SIZE, cfg.HR_SIZE), \
            f"Generator output shape wrong: {out.shape}"
    print(f"Generator sanity check passed ✓  output {out.shape}")

    # ── Losses ─────────────────────────────────────────────
    l1_fn   = nn.L1Loss()
    gan_fn  = GANLoss(smooth=0.1)
    perc_fn = None
    if cfg.USE_PERCEPTUAL:
        perc_fn = PerceptualLoss(str(VGG_WEIGHTS_PATH)
                                 if VGG_WEIGHTS_PATH else None).to(cfg.DEVICE)
        perc_fn.eval()

    # ── Optimisers ─────────────────────────────────────────
    opt_g = optim.Adam(G.parameters(), lr=cfg.LR_G,
                       betas=(cfg.BETA1, cfg.BETA2))
    opt_d = optim.Adam(D.parameters(), lr=cfg.LR_D,
                       betas=(cfg.BETA1, cfg.BETA2))

    sched_g = optim.lr_scheduler.CosineAnnealingLR(
        opt_g, T_max=cfg.NUM_EPOCHS, eta_min=1e-6)
    sched_d = optim.lr_scheduler.CosineAnnealingLR(
        opt_d, T_max=cfg.NUM_EPOCHS, eta_min=1e-6)

    # ── AMP ────────────────────────────────────────────────
    use_amp = cfg.MIXED_PRECISION and pin   # only on CUDA
    scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)

    # ── Resume ─────────────────────────────────────────────
    start_ep  = 0
    best_mae  = float('inf')
    ckpt_path = os.path.join(cfg.CHECKPOINT_DIR, 'latest.pth')
    if os.path.exists(ckpt_path):
        start_ep = load_ckpt(ckpt_path, G, D, opt_g, opt_d, cfg.DEVICE) + 1

    history = {k: [] for k in ['g', 'd', 'val_mae', 'val_psnr']}

    for epoch in range(start_ep, cfg.NUM_EPOCHS):
        G.train(); D.train()
        m = {k: AverageMeter() for k in ['g','d','l1','adv','pc']}

        bar = tqdm(train_loader,
                   desc=f"Ep {epoch+1:3d}/{cfg.NUM_EPOCHS}",
                   leave=False, dynamic_ncols=True)

        for lr_t, hr_t in bar:
            lr_t = lr_t.to(cfg.DEVICE, non_blocking=True)
            hr_t = hr_t.to(cfg.DEVICE, non_blocking=True)
            bs   = lr_t.size(0)

            # ── Discriminator ──────────────────────────────
            opt_d.zero_grad(set_to_none=True)
            with amp_autocast(use_amp):
                fake_d   = G(lr_t).detach()
                r_pred   = D(hr_t)
                f_pred   = D(fake_d)
                d_loss   = 0.5 * (gan_fn(r_pred, True) + gan_fn(f_pred, False))
            scaler.scale(d_loss).backward()
            scaler.step(opt_d)

            # ── Generator ──────────────────────────────────
            opt_g.zero_grad(set_to_none=True)
            with amp_autocast(use_amp):
                fake_g  = G(lr_t)
                gp      = D(fake_g)
                l_l1    = l1_fn(fake_g, hr_t)         * cfg.LAMBDA_L1
                l_adv   = gan_fn(gp, True)             * cfg.LAMBDA_ADV
                l_pc    = (perc_fn(fake_g, hr_t)       * cfg.LAMBDA_PERC
                           if perc_fn else
                           torch.tensor(0., device=cfg.DEVICE))
                g_loss  = l_l1 + l_adv + l_pc
            scaler.scale(g_loss).backward()
            scaler.step(opt_g)
            scaler.update()

            m['g'].update(g_loss.item(),  bs)
            m['d'].update(d_loss.item(),  bs)
            m['l1'].update(l_l1.item(),   bs)
            m['adv'].update(l_adv.item(), bs)
            m['pc'].update(l_pc.item(),   bs)
            bar.set_postfix(G=f"{m['g'].avg:.3f}",
                            D=f"{m['d'].avg:.3f}",
                            L1=f"{m['l1'].avg:.3f}")

        sched_g.step(); sched_d.step()

        # ── Validation ─────────────────────────────────────
        G.eval()
        vm = {k: AverageMeter() for k in ['mae','psnr']}
        with torch.no_grad():
            for lr_t, hr_t in val_loader:
                lr_t = lr_t.to(cfg.DEVICE); hr_t = hr_t.to(cfg.DEVICE)
                with amp_autocast(use_amp):
                    fake = G(lr_t)
                f255 = ((fake  + 1.)/2.*255.).clamp(0,255)
                r255 = ((hr_t  + 1.)/2.*255.).clamp(0,255)
                vm['mae'].update(F.l1_loss(f255, r255).item(), lr_t.size(0))
                vm['psnr'].update(psnr(fake, hr_t).item(), lr_t.size(0))

        history['g'].append(m['g'].avg)
        history['d'].append(m['d'].avg)
        history['val_mae'].append(vm['mae'].avg)
        history['val_psnr'].append(vm['psnr'].avg)

        star = ""
        if vm['mae'].avg < best_mae:
            best_mae = vm['mae'].avg; star = " ★"
            save_ckpt(dict(epoch=epoch, generator=G.state_dict(),
                           discriminator=D.state_dict(),
                           opt_g=opt_g.state_dict(), opt_d=opt_d.state_dict(),
                           val_mae=best_mae),
                      os.path.join(cfg.CHECKPOINT_DIR, 'best.pth'))

        print(f"[{epoch+1:3d}/{cfg.NUM_EPOCHS}]  "
              f"G={m['g'].avg:.4f}  L1={m['l1'].avg:.4f}  "
              f"D={m['d'].avg:.4f}  "
              f"ValMAE={vm['mae'].avg:.4f}  "
              f"PSNR={vm['psnr'].avg:.2f}dB{star}")

        save_ckpt(dict(epoch=epoch, generator=G.state_dict(),
                       discriminator=D.state_dict(),
                       opt_g=opt_g.state_dict(), opt_d=opt_d.state_dict(),
                       val_mae=vm['mae'].avg),
                  ckpt_path)

        if (epoch + 1) % cfg.SAVE_EVERY == 0:
            save_ckpt(dict(epoch=epoch, generator=G.state_dict(),
                           discriminator=D.state_dict(),
                           opt_g=opt_g.state_dict(), opt_d=opt_d.state_dict(),
                           val_mae=vm['mae'].avg),
                      os.path.join(cfg.CHECKPOINT_DIR, f'ep{epoch+1:04d}.pth'))

        gc.collect()

    print(f"\nTraining done — best Val MAE = {best_mae:.4f}")
    return G, history, va_lr, va_hr


# ============================================================
# 11. VISUALISATION
# ============================================================
def visualize(G, va_lr, va_hr, n=4):
    G.eval()
    n   = min(n, len(va_lr))
    ids = random.sample(range(len(va_lr)), n)
    ds  = SRDataset(va_lr, va_hr, augment=False)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
    if n == 1: axes = axes[None]

    with torch.no_grad():
        for row, idx in enumerate(ids):
            lr_t, hr_t = ds[idx]
            sr_t = G(lr_t.unsqueeze(0).to(cfg.DEVICE)).squeeze(0).cpu()
            lr_np = tensor_to_uint8(lr_t)
            sr_np = tensor_to_uint8(sr_t)
            hr_np = tensor_to_uint8(hr_t)
            bic   = np.array(Image.fromarray(lr_np).resize(
                        (cfg.HR_SIZE, cfg.HR_SIZE), Image.BICUBIC))
            for col, (img, ttl) in enumerate([
                    (bic,   'Bicubic ↑'),
                    (sr_np, 'SR (ours)'),
                    (hr_np, 'Ground Truth')]):
                axes[row,col].imshow(img)
                axes[row,col].set_title(ttl)
                axes[row,col].axis('off')

    plt.tight_layout()
    p = os.path.join(cfg.OUTPUT_DIR, 'comparison.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"Comparison saved → {p}")


def plot_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history['g'], label='G'); axes[0].plot(history['d'], label='D')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(history['val_mae'],  color='red');   axes[1].set_title('Val MAE ↓');  axes[1].grid(True)
    axes[2].plot(history['val_psnr'], color='green'); axes[2].set_title('Val PSNR ↑'); axes[2].grid(True)
    plt.tight_layout()
    p = os.path.join(cfg.OUTPUT_DIR, 'history.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"History saved → {p}")


# ============================================================
# 12. INFERENCE + SUBMISSION
# ============================================================
def generate_submission(G):
    print("\n" + "=" * 60)
    print("INFERENCE + SUBMISSION")
    print("=" * 60)

    best = os.path.join(cfg.CHECKPOINT_DIR, 'best.pth')
    if os.path.exists(best):
        c = torch.load(best, map_location=cfg.DEVICE)
        G.load_state_dict(c['generator'])
        print(f"Best checkpoint loaded — epoch {c['epoch']+1}  MAE={c['val_mae']:.4f}")

    G.eval()
    use_amp = cfg.MIXED_PRECISION and (cfg.DEVICE.type == 'cuda')

    test_ds = SRDataset(test_lr_files, is_test=True)
    loader  = DataLoader(test_ds, batch_size=8, shuffle=False,
                         num_workers=cfg.NUM_WORKERS,
                         pin_memory=(cfg.DEVICE.type == 'cuda'))

    EXPECTED = cfg.HR_SIZE * cfg.HR_SIZE * cfg.CHANNELS   # 49 152
    rows = []

    with torch.no_grad():
        for lr_b, fnames in tqdm(loader, desc="Inference"):
            lr_b = lr_b.to(cfg.DEVICE)
            with amp_autocast(use_amp):
                sr_b = G(lr_b)

            for i, fname in enumerate(fnames):
                sr = sr_b[i]                              # (3,128,128)
                # [-1,1] → [0,255] uint8
                sr_np = ((sr + 1.0) / 2.0).clamp(0, 1)
                sr_np = sr_np.cpu().float().numpy()       # (3,H,W)
                sr_np = (sr_np * 255.0).round().astype(np.uint8)
                # (3,H,W) → (H,W,3) → flatten (row-major per-pixel RGB)
                flat  = sr_np.transpose(1, 2, 0).flatten()
                assert len(flat) == EXPECTED, \
                    f"{fname}: {len(flat)} px ≠ {EXPECTED}"
                rows.append({'Id': fname, 'Pixels': ' '.join(map(str, flat.tolist()))})

    df = (pd.DataFrame(rows, columns=['Id','Pixels'])
            .sort_values('Id').reset_index(drop=True))
    df.to_csv(cfg.SUBMISSION_PATH, index=False)

    print(f"Saved → {cfg.SUBMISSION_PATH}  ({len(df)} rows)")
    print(f"Sample : {df['Id'].iloc[0]}  |  first 20 px: "
          f"{' '.join(df['Pixels'].iloc[0].split()[:20])} …")
    return df


# ============================================================
# 13. VALIDATION
# ============================================================
def validate_submission(df):
    EXPECTED = cfg.HR_SIZE * cfg.HR_SIZE * cfg.CHANNELS
    errors = []
    for i, row in df.iterrows():
        toks = row['Pixels'].split()
        if len(toks) != EXPECTED:
            errors.append(f"Row {i}: {len(toks)} px"); continue
        try:
            vals = list(map(int, toks))
        except ValueError as e:
            errors.append(f"Row {i}: non-int — {e}"); continue
        if min(vals) < 0 or max(vals) > 255:
            errors.append(f"Row {i}: out-of-range")

    if errors:
        print(f"\n{len(errors)} error(s):"); [print(f"  {e}") for e in errors[:5]]
    else:
        print(f"\n✓ All {len(df)} rows valid — {EXPECTED} int px each in [0,255]")


# ============================================================
# 14. MAIN
# ============================================================
G, history, va_lr, va_hr = train()

try:
    visualize(G, va_lr, va_hr, n=4)
except Exception as e:
    print(f"Visualisation skipped: {e}")

plot_history(history)

df_sub = generate_submission(G)
validate_submission(df_sub)

print(f"\n✓ Submission ready → {cfg.SUBMISSION_PATH}")